# 1주차 7교시 — 결측치·스케일링·인코딩 기준을 train에서만 배우기

## 학생용 지적 실험 노트

**수업이 끝나면 할 수 있는 것:** 숫자와 문자 열을 나누어 결측치·크기·범주를 처리하고 test에는 transform만 적용합니다.

> 실행 전에 반드시 예상 답을 적습니다. `___` 또는 `FILL_ME`만 채운 뒤 실행하고, 값을 한 번 바꾸고, 결과가 왜 달라졌는지 한 문장으로 적습니다.


## 지금 열 파일과 수업 진행 방법

- 지금 열 파일: `1주차_7교시_학생용_지적실험노트.ipynb`
- 함께 받은 `matchmaking_data/01_week1` 폴더의 CSV를 사용합니다. 폴더 위치는 바꾸지 않습니다.
- 강사가 PPT에서 Q번호를 보여주면 노트북에서 같은 Q번호를 찾습니다.
- 먼저 결과를 예상한 뒤 빈칸을 채우고 실행합니다.
- 교시가 끝나면 자신이 작성한 노트북을 저장합니다.



## 50분 운영 리듬

- 0~5분 · fit과 transform 회상
- 5~13분 · 숫자 열과 문자 열 나누기
- 13~22분 · SimpleImputer
- 22~30분 · StandardScaler
- 30~37분 · OneHotEncoder
- 37~46분 · ColumnTransformer·Pipeline
- 46~50분 · 누수 복구·새 범주가 있는 데이터에 적용


## 준비 셀 — 데이터와 라이브러리 연결

- **뜻:** 오늘 사용할 도구와 데이터 폴더를 준비합니다.
- **이유:** 뒤의 Q가 데이터 위치와 앞 교시에 의존하지 않게 만듭니다.
- **정상:** 데이터 폴더 경로가 출력됩니다.
- **복구:** 프로젝트 폴더 안에서 노트북을 열고 이 셀부터 다시 실행합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_week1_data_dir():
    """어디에서 노트북을 열어도 수업 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "01_week1",
            base / "matchmaking_data" / "01_week1",
            base / "01_week1",
        ]
        for candidate in candidates:
            if (candidate / "matchmaking_members.csv").exists():
                return candidate
    raise FileNotFoundError("matchmaking_members.csv를 찾지 못했습니다. 프로젝트 폴더 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week1_data_dir()
print("데이터 폴더:", DATA_DIR)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


## 수업에서 사용할 데이터 준비

이 셀은 미리 제공하는 준비 코드입니다. 학생은 따라 치지 않고 실행 결과의 행·열 수만 확인합니다.


In [ ]:
members = pd.read_csv(DATA_DIR / "matchmaking_members.csv").drop_duplicates("member_id", keep="first").copy()
preferences = pd.read_csv(DATA_DIR / "matchmaking_preferences.csv")
members["residence_region"] = members["residence_region"].str.strip().replace({"Seoul":"서울", "seoul":"서울"})
members["annual_income_10k_krw"] = pd.to_numeric(members["annual_income_10k_krw"], errors="coerce")
joined = pd.merge(members, preferences, on="member_id", how="left", validate="one_to_one")

numeric_features = ["age", "annual_income_10k_krw", "profile_completion_rate", "recommendation_response_rate_prior", "consultation_count_prior", "pref_income_min_10k_krw", "preference_strictness"]
categorical_features = ["gender", "residence_region", "occupation_type", "membership_tier", "signup_channel", "pref_education_min", "deal_breakers"]
X = joined[numeric_features + categorical_features].copy()
y = joined["meeting_scheduled"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)


## Q1. 숫자와 문자 열은 왜 서로 다른 처리 방법이 필요할까요?

> **처음 배우는 사람을 위한 안내**
>
> 뜻: select_dtypes로 계산 가능한 숫자와 범주 의미의 문자를 구분합니다.
> 
> 이유: 문자에 중앙값·표준화를 하거나 숫자를 수백 개 범주로 펼치는 실수를 막습니다.
> 
> 정상: 수치형 7열, 범주형 7열입니다.
> 
> 복구: X_train.dtypes와 미리 정한 feature 목록을 함께 확인합니다.

### 실행 전에 생각할 질문

열 이름을 외우기보다 ‘값 사이의 크기 차이가 의미 있나?’로 숫자·범주 처리를 구분합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
detected_numeric = X_train.___(include="number").columns.tolist()
detected_categorical = X_train.___(exclude="number").columns.tolist()
print("숫자:", detected_numeric)
print("문자:", detected_categorical)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q2. SimpleImputer는 어떤 기준을 어디에서 배워야 할까요?

> **처음 배우는 사람을 위한 안내**
>
> 뜻: fit은 train 중앙값을 저장하고 transform은 저장된 값으로 빈칸을 채웁니다.
> 
> 이유: test의 정보를 미리 본 중앙값을 쓰는 데이터 누수를 막습니다.
> 
> 정상: income 중앙값이 statistics_에 있고 train/test 변환 결과 결측이 0입니다.
> 
> 복구: test에 fit_transform을 쓰지 않았는지, 2차원 DataFrame을 넘겼는지 확인합니다.

### 실행 전에 생각할 질문

fit=처리 기준 계산·저장, transform=그 기준 적용이라고 소리 내어 구분합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
income_imputer = SimpleImputer(strategy=___)
train_income = income_imputer.___(X_train[["annual_income_10k_krw"]])
test_income = income_imputer.___(X_test[["annual_income_10k_krw"]])
print("배운 중앙값:", income_imputer.statistics_)
print("남은 결측:", np.isnan(train_income).sum(), np.isnan(test_income).sum())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q3. StandardScaler는 단위를 지우고 상대적 위치를 남깁니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: train 평균을 0, 표준편차를 1에 가깝게 바꾸는 기준을 학습합니다.
> 
> 이유: 연봉처럼 큰 단위가 나이 같은 작은 단위를 숫자 크기만으로 압도하지 않게 합니다.
> 
> 정상: train 두 열의 평균이 거의 0입니다. test는 꼭 0일 필요가 없습니다.
> 
> 복구: 결측치를 먼저 채우고 scaler도 train에 fit, test에 transform만 합니다.

### 실행 전에 생각할 질문

스케일링은 중요도를 동일하게 만드는 것이 아니라 숫자 단위의 출발선을 맞추는 것입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
two_numeric = ["age", "annual_income_10k_krw"]
imputer2 = SimpleImputer(strategy="median")
train_two_filled = imputer2.fit_transform(X_train[two_numeric])
test_two_filled = imputer2.transform(X_test[two_numeric])

scaler = StandardScaler()
train_two_scaled = scaler.___(train_two_filled)
test_two_scaled = scaler.___(test_two_filled)
print("train 평균:", train_two_scaled.mean(axis=0))


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q4. OneHotEncoder는 문자 범주를 순서 없는 0·1 열로 바꿉니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: 각 범주마다 열을 만들고 해당하면 1, 아니면 0을 둡니다.
> 
> 이유: 서울=1, 부산=2처럼 가짜 크기 관계를 만들지 않고 계산 가능하게 합니다.
> 
> 정상: region 범주 수만큼 열이 생기고 test의 처음 보는 범주도 오류 없이 처리됩니다.
> 
> 복구: handle_unknown='ignore'를 확인하고 fit은 train에만 합니다.

### 실행 전에 생각할 질문

새 회원 파일의 처음 보는 범주가 와도 오류 없이 처리하게 하는 설정이 handle_unknown입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
region_encoder = OneHotEncoder(handle_unknown=___, sparse_output=False)
train_region = region_encoder.___(X_train[["residence_region"]])
test_region = region_encoder.___(X_test[["residence_region"]])
print(region_encoder.get_feature_names_out())
print(train_region.shape, test_region.shape)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q5. Pipeline은 처리 순서를, ColumnTransformer는 열을 나누는 기준을 묶습니다

> **처음 배우는 사람을 위한 안내**
>
> 뜻: 숫자는 중앙값→표준화, 문자는 최빈값→One-Hot으로 보내는 하나의 전처리기를 만듭니다.
> 
> 이유: train과 새 데이터에 같은 열·같은 순서·같은 기준을 반복 적용하기 위해서입니다.
> 
> 정상: preprocessor 출력에 num·cat 두 분기와 각 단계가 보입니다.
> 
> 복구: Pipeline 튜플의 이름·객체 순서와 ColumnTransformer의 열 목록을 확인합니다.

### 실행 전에 생각할 질문

Pipeline은 한 종류의 열에 적용할 단계 순서이고, ColumnTransformer는 숫자 열과 문자 열에 서로 다른 처리를 적용한 뒤 결과를 합친다고 설명합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy=___)),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy=___)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, ___),
    ("cat", categorical_pipe, ___)
], verbose_feature_names_out=False)
print(preprocessor)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q6. train에 fit_transform, test에는 transform만 실행하세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: train에서 모든 대체값·평균·표준편차·범주 목록을 배우고 test에는 고정 적용합니다.
> 
> 이유: 평가 자료를 미리 보지 않으면서도 두 결과의 열 구조를 동일하게 만들기 위해서입니다.
> 
> 정상: 800행/200행, 열 수 동일, 결측치 0입니다.
> 
> 복구: NotFittedError면 train 셀부터, 열 수가 다르면 test에 새 encoder를 fit했는지 확인합니다.

### 실행 전에 생각할 질문

이 셀이 7교시의 핵심 원칙입니다. 코드보다 ‘어디에 fit했나’를 먼저 표시합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
train_processed = preprocessor.___(X_train)
test_processed = preprocessor.___(X_test)
print(train_processed.shape, test_processed.shape)
print("남은 결측:", np.isnan(train_processed).sum(), np.isnan(test_processed).sum())
assert train_processed.shape[1] == test_processed.shape[1]


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q7. 코드 골격을 완성해 새 회원 100명에 같은 기준 적용하기

> **처음 배우는 사람을 위한 안내**
>
> 성공 조건: new CSV를 같은 14개 열로 준비하고 preprocessor.transform만 실행합니다. fit 또는 fit_transform을 쓰면 실패입니다.
> 
> 빈칸 안내: 파일 읽기, 지역 문자열 정리, 연봉 숫자 변환과 기존 처리기 적용에 필요한 기능을 채웁니다. 14개 열을 맞추는 코드는 제공합니다.
> 
> 정상: 100행, train과 같은 열 수, 결측치 0입니다. 새 가입경로 ‘웨비나’가 있어도 오류가 나지 않습니다.
> 
> 복구: 열 순서·이름을 X.columns와 맞추고 handle_unknown='ignore'를 확인합니다.

### 실행 전에 생각할 질문

새 데이터는 시험지이자 서비스 입력입니다. 기준을 다시 배우지 않고 기존 기준을 재사용합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
# 1. 새 회원 파일을 읽습니다.
new_members = pd.___(
    DATA_DIR / "matchmaking_members_new.csv"
).copy()

# 2. 지역 양쪽의 불필요한 공백과 영문 표기를 정리합니다.
new_members["residence_region"] = (
    new_members["residence_region"].str.___()
    .replace({"Seoul": "서울", "seoul": "서울"})
)

# 3. 연봉을 계산 가능한 숫자로 바꿉니다.
new_members["annual_income_10k_krw"] = pd.___(
    new_members["annual_income_10k_krw"], errors="coerce"
)

# 4. 신규회원 파일에 없는 선호조건 열은 이번 실습용 대표값으로 준비합니다.
new_members["pref_income_min_10k_krw"] = np.nan
new_members["preference_strictness"] = X_train["preference_strictness"].median()
new_members["pref_education_min"] = X_train["pref_education_min"].mode().iloc[0]
new_members["deal_breakers"] = np.nan
new_X = new_members[numeric_features + categorical_features].copy()

# 5. 기준을 다시 배우지 않고, train에서 배운 처리기를 그대로 적용합니다.
new_processed = preprocessor.___(new_X)

print(new_processed.shape)
print("남은 결측:", np.isnan(new_processed).sum())
assert new_processed.shape[0] == 100
assert new_processed.shape[1] == train_processed.shape[1]
assert np.isnan(new_processed).sum() == 0


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## 마무리 질문

1. 오늘 가장 자주 쓸 기능 하나와 그 이유는?
2. 오늘 기능을 쓰게 되는 데이터의 신호는?
3. 정상 결과를 무엇으로 확인했나?
4. 다음 교시로 넘겨야 할 결과는?
